# 5章 構造化データの出力 (3)

## 事前準備

当サンプルコードを実行するには、OpenAIのキーが必要です。

### OpenAI
- OpenAIのプラットフォーム（ https://platform.openai.com/ ）にアクセスし、ご自身のアカウントでログインして、APIキーを生成してください。
- 生成したAPIキーを定数 `OPENAI_API_KEY` に設定します。

```
OPENAI_API_KEY=<作成したAPIキー>
```



In [ ]:
OPENAI_API_KEY= "<作成したAPIキー>"

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

### LangChain

In [ ]:
!wget "https://drive.google.com/uc?export=download&id=1QQlSIaCDIFql-IHBd9NHd3YiqTAN2gr_" -O apple.txt

--2026-10-22 06:19:47--  https://drive.google.com/uc?export=download&id=1QQlSIaCDIFql-IHBd9NHd3YiqTAN2gr_
Resolving drive.google.com (drive.google.com)... 172.217.204.138, 172.217.204.113, 172.217.204.101, ...
Connecting to drive.google.com (drive.google.com)|172.217.204.138|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1QQlSIaCDIFql-IHBd9NHd3YiqTAN2gr_&export=download [following]
--2026-10-22 06:19:47--  https://drive.usercontent.google.com/download?id=1QQlSIaCDIFql-IHBd9NHd3YiqTAN2gr_&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 74.125.196.132, 2607:f8b0:400c:c3f::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|74.125.196.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 410321 (401K) [application/octet-stream]
Saving to: ‘apple.txt’

apple.txt           100%[===================>] 400.70K  --.-KB/s  

In [ ]:
MAX_LENGTH = 100_000
# サンプルの10-K SEC提出書類からテキストを読み込む
with open('apple.txt', 'r') as file:
    sec_filing = file.read()
sec_filing = sec_filing[:MAX_LENGTH]

In [ ]:
!pip install langchain_openai langchain_anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.5/120.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 36.6 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.8
    Uninstalling langchain-core-1.4.8:
      Successfully uninstalled langchain-core-1.4.8


In [ ]:
from pydantic import BaseModel
from openai import OpenAI

class SECExtraction(BaseModel):
    mentioned_entities: list[str]
    mentioned_places: list[str]

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate

def extract_from_sec_filing_langchain(
    sec_filing_text: str,
    prompt: str
) -> SECExtraction:
    """
    LangChainを使用して、入力されたSEC提出書類のテキストから構造化データを抽出する。
    """
    llm_openai = ChatOpenAI(model="gpt-5.4-mini")

    structured_llm = llm_openai.with_structured_output(SECExtraction)

    prompt_template = ChatPromptTemplate.from_messages(
        [
            ("system", prompt),
            ("human", "{sec_filing_text}"),
        ]
    )

    llm_chain = prompt_template | structured_llm

    return llm_chain.invoke(sec_filing_text)

In [ ]:
prompt_extraction = '''あなたは構造化データ抽出の専門家です。
SEC提出書類から得た非構造化テキストが与えられるので、
そこに記載されている企業・組織などのエンティティ名と場所を、
指定された構造に抽出してください。'''

sec_extraction_langchain = extract_from_sec_filing_langchain(
    sec_filing,
    prompt_extraction
)

In [ ]:
print("Extracted entities:", sec_extraction_langchain.mentioned_entities)
print("Extracted places:", sec_extraction_langchain.mentioned_places)

Extracted entities: ['Apple Inc.', 'Apple', 'AppleCare', 'App Store', 'Apple Arcade', 'Apple Fitness+', 'Apple Music', 'Apple News+', 'Apple TV+', 'Apple Card', 'Apple Pay', 'Apple Watch Ultra 2', 'Apple Watch Series 10', 'Apple Watch SE', 'AirPods', 'AirPods Pro', 'AirPods Max', 'Beats', 'Apple Vision Pro', 'Apple TV', 'HomePod', 'HomePod mini', 'Mac', 'MacBook Air', 'MacBook Pro', 'iMac', 'Mac mini', 'Mac Studio', 'Mac Pro', 'iPhone', 'iPad', 'iPad Pro', 'iPad Air', 'iPad mini', 'Apple University', 'U.S. Securities and Exchange Commission', 'SEC', 'European Commission', 'General Court of the EU', 'U.S. Department of Justice', 'DOJ', 'Epic Games, Inc.', 'Google LLC', 'The Nasdaq Stock Market LLC', 'S&P 500 Index', 'Dow Jones U.S. Technology Supersector Index', 'Organisation for Economic Co-operation and Development', 'Internal Revenue Service']
Extracted places: ['Cupertino, California', 'U.S.', 'China mainland', 'Hong Kong', 'Taiwan', 'India', 'Middle East', 'Africa', 'Europe', 'Japa